# SENTINEL — Multi-Agent RL for Cloud Incident Response
### OpenEnv Hackathon 2026 | Meta PyTorch

This notebook runs the **full SENTINEL pipeline** top-to-bottom:

| Cell | What it does |
|------|-------------|
| 1 | Install dependencies |
| 2 | Instantiate environment, print observation/action spaces |
| 3 | **Baseline** — random actions, 30 episodes |
| 4 | **Trained** — UCB1 bandit + Bayesian RCA, 30 episodes |
| 5 | Plot reward curves + save PNG |
| 6 | Before/After behavior transcript |
| 7 | Summary table |

**Mathematical algorithms used (no LLM, no API keys needed):**
- UCB1 multi-armed bandit (Auer et al. 2002) for action selection
- Bayesian Noisy-OR (Pearl 1988 / MicroRank WWW 2021) for root cause analysis
- Personalized PageRank (Brin & Page 1998) for remediation ranking
- ALP Curriculum (Portelas et al. CoRL 2020) for Oracle scenario generation

> **No GPU required.** All cells run on CPU.

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys, os

# Clone repo if running in Colab
if not os.path.exists('sentinel'):
    subprocess.run(['git', 'clone', 'https://github.com/SayantikaLaskar/sentinel.git'], check=False)
    os.chdir('sentinel')
elif os.path.basename(os.getcwd()) != 'sentinel':
    os.chdir('sentinel')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Dependencies installed')

## Cell 1b — Pull Latest Fixes

In [ ]:
import subprocess
# Pull latest fixes (action parser + prompt builder + llm agent improvements)
result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('git pull failed:', result.stderr)
else:
    print('Code is up to date')

## Cell 2 — Instantiate Environment & Verify

In [ ]:
import sys, os
if os.path.exists('sentinel') and '.' not in sys.path:
    sys.path.insert(0, '.')

from sentinel.env import Sentinel_Env

env = Sentinel_Env(
    config_path='env_spec.yaml',
    incident_library_path='incident_library.yaml',
    render_mode='human'
)

obs, info = env.reset(seed=42)
print('Environment created')
print(f'Incident injected: {info["incident_id"]}')
print()
print(env.render())
print()
print('Observation keys:', list(obs.keys()))
print('Observation space:', env.observation_space)
print('Action space:     ', env.action_space)

## Cell 3 — Baseline (Random Actions) — 30 Episodes
Random actions with no intelligence. This is the floor that training improves upon.

In [ ]:
import random
from sentinel.env import Sentinel_Env

SEED = 0
N_EPISODES = 30
random.seed(SEED)

baseline_env = Sentinel_Env(
    config_path='env_spec.yaml',
    incident_library_path='incident_library.yaml'
)

# Random action pool — no intelligence, just cycles through valid actions
RANDOM_ACTIONS = [
    {'agent': 'holmes', 'category': 'investigative', 'name': 'QueryLogs',
     'params': {'service': 'api-gateway', 'time_range': [0, 60]}},
    {'agent': 'holmes', 'category': 'investigative', 'name': 'QueryMetrics',
     'params': {'service': 'web-gateway', 'metric_name': 'cpu', 'time_range': [0, 300]}},
    {'agent': 'forge', 'category': 'remediation', 'name': 'RestartService',
     'params': {'service': 'api-gateway'}},
    {'agent': 'forge', 'category': 'remediation', 'name': 'ScaleService',
     'params': {'service': 'web-gateway', 'replicas': 2}},
]

baseline_rewards = []
for ep in range(N_EPISODES):
    obs, info = baseline_env.reset(seed=SEED + ep)
    terminated = truncated = False
    ep_reward = 0.0
    step = 0
    while not (terminated or truncated) and step < 60:
        action = RANDOM_ACTIONS[step % len(RANDOM_ACTIONS)]
        obs, r, terminated, truncated, _ = baseline_env.step(action)
        ep_reward += r
        step += 1
    baseline_rewards.append(ep_reward)
    if (ep + 1) % 10 == 0:
        avg = sum(baseline_rewards[-10:]) / 10
        print(f'Episode {ep+1:3d} | avg(last 10) reward = {avg:.4f}')

print(f'\nBaseline complete — mean reward: {sum(baseline_rewards)/len(baseline_rewards):.4f}')

## Cell 4 — Trained Agent (UCB1 + Bayesian RCA) — 30 Episodes

Uses the **mathematical intelligence layer** (`sentinel/math_engine.py`):
- **UCB1 bandit** (Auer et al. 2002) selects which action type to try
- **Bayesian Noisy-OR** (Pearl 1988) identifies the most likely root-cause service
- **Personalized PageRank** (MicroRank WWW 2021) ranks remediation targets

No LLM, no API keys — pure math.

In [ ]:
from sentinel.env import Sentinel_Env
from sentinel.training.pipeline import _get_action, TrainingConfig
from sentinel.math_engine import get_ucb1_bandit

SEED = 100
N_EPISODES = 30

trained_env = Sentinel_Env(
    config_path='env_spec.yaml',
    incident_library_path='incident_library.yaml'
)
cfg = TrainingConfig(agent='holmes')
bandit = get_ucb1_bandit()

trained_rewards = []
for ep in range(N_EPISODES):
    obs, info = trained_env.reset(seed=SEED + ep)
    terminated = truncated = False
    ep_reward = 0.0
    step = 0
    while not (terminated or truncated) and step < 60:
        action = _get_action(None, obs, cfg)
        arm_idx = action.pop('_ucb1_arm_idx', None)
        obs, r, terminated, truncated, _ = trained_env.step(action)
        if arm_idx is not None:
            bandit.update(arm_idx, float(r))
        ep_reward += r
        step += 1
    trained_rewards.append(ep_reward)
    if (ep + 1) % 10 == 0:
        avg = sum(trained_rewards[-10:]) / 10
        print(f'Episode {ep+1:3d} | avg(last 10) reward = {avg:.4f}')

print(f'\nTrained complete — mean reward: {sum(trained_rewards)/len(trained_rewards):.4f}')

# Print UCB1 arm statistics
print('\nUCB1 Top Arms (Auer 2002):')
for s in bandit.arm_stats()[:5]:
    print(f'  {s["arm"]}: pulls={s["pulls"]}, mean_reward={s["mean_reward"]}')

## Cell 5 — Plot Training Curves & Save to results/

In [ ]:
import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

os.makedirs('results', exist_ok=True)

def smooth(values, window=5):
    if len(values) < window:
        return values
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode='valid').tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f172a')
for ax in axes:
    ax.set_facecolor('#1e293b')
    ax.tick_params(colors='#94a3b8')
    for spine in ax.spines.values():
        spine.set_color('#334155')

# Left: episode reward comparison
ax = axes[0]
x_b = range(len(smooth(baseline_rewards)))
x_t = range(len(smooth(trained_rewards)))
ax.plot(x_b, smooth(baseline_rewards), color='#ef4444', linewidth=2, label='Baseline (random)')
ax.plot(x_t, smooth(trained_rewards),  color='#22c55e', linewidth=2, label='UCB1+Bayesian RCA')
ax.axhline(0, color='#475569', linestyle='--', linewidth=0.8)
ax.set_title('Episode Reward (smoothed)', color='#e2e8f0', fontsize=13)
ax.set_xlabel('Episode', color='#94a3b8')
ax.set_ylabel('Total Reward', color='#94a3b8')
ax.legend(facecolor='#1e293b', edgecolor='#334155', labelcolor='#e2e8f0')

# Right: bar chart comparing means
ax = axes[1]
categories = ['Baseline\n(random)', 'Trained\n(UCB1+Bayes)']
means = [sum(baseline_rewards)/len(baseline_rewards), sum(trained_rewards)/len(trained_rewards)]
colors = ['#ef4444', '#22c55e']
bars = ax.bar(categories, means, color=colors, edgecolor='#e2e8f0', linewidth=0.5, width=0.5)
for bar, val in zip(bars, means):
    y = bar.get_height()
    offset = 0.02 if y >= 0 else -0.06
    ax.text(bar.get_x() + bar.get_width()/2, y + offset,
            f'{val:.3f}', ha='center', va='bottom', color='#e2e8f0', fontsize=12, fontweight='bold')
ax.set_title('Mean Episode Reward', color='#e2e8f0', fontsize=13)
ax.set_ylabel('Reward', color='#94a3b8')
ax.axhline(0, color='#475569', linestyle='--', linewidth=0.8)

fig.suptitle('SENTINEL — Training Results', color='#f8fafc', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
print('Plot saved to results/training_curves.png')
plt.show()

## Cell 6 — Before/After Behavior Transcript

In [ ]:
import json, os
from sentinel.env import Sentinel_Env
from sentinel.training.pipeline import _get_action, TrainingConfig

TRANSCRIPT_SEED = 77

def capture_transcript(env, action_fn, label, n_steps=10):
    lines = [f'\n{"="*60}', f'  {label}', f'{"="*60}']
    obs, info = env.reset(seed=TRANSCRIPT_SEED)
    iid = info.get('incident_id', 'unknown')
    lines.append(f'Incident: {iid}')
    rendered = env.render()
    if rendered:
        lines.append(rendered)
    lines.append('')
    terminated = truncated = False
    step = 0
    cumulative = 0.0
    while not (terminated or truncated) and step < n_steps:
        action = action_fn(obs, step)
        obs, r, terminated, truncated, info = env.step(action)
        cumulative += r
        agent = action.get('agent', '?')
        name = action.get('name', '?')
        svc = action.get('params', {}).get('service', '-')
        lines.append(f'  Step {step+1:2d} | {agent:8s} | {name:20s} | target={svc:20s} | r={r:+.3f} | cum={cumulative:+.3f}')
        step += 1
    lines.append(f'\n  Final reward after {step} steps: {cumulative:+.4f}')
    return '\n'.join(lines)

# BEFORE: random cycling actions (no intelligence)
RANDOM_ACTIONS = [
    {'agent': 'holmes', 'category': 'investigative', 'name': 'QueryLogs',
     'params': {'service': 'api-gateway', 'time_range': [0, 60]}},
    {'agent': 'forge', 'category': 'remediation', 'name': 'RestartService',
     'params': {'service': 'api-gateway'}},
]
before_env = Sentinel_Env(config_path='env_spec.yaml', incident_library_path='incident_library.yaml', render_mode='human')
before_txt = capture_transcript(
    before_env,
    lambda obs, step: RANDOM_ACTIONS[step % len(RANDOM_ACTIONS)],
    'BEFORE TRAINING (Random Actions — no intelligence)'
)

# AFTER: UCB1 + Bayesian RCA
after_env = Sentinel_Env(config_path='env_spec.yaml', incident_library_path='incident_library.yaml', render_mode='human')
after_cfg = TrainingConfig(agent='holmes')
def trained_action_fn(obs, step):
    action = _get_action(None, obs, after_cfg)
    action.pop('_ucb1_arm_idx', None)
    return action
after_txt = capture_transcript(
    after_env,
    trained_action_fn,
    'AFTER TRAINING (UCB1 Bandit + Bayesian RCA)'
)

transcript = before_txt + '\n\n' + after_txt
print(transcript)

os.makedirs('results', exist_ok=True)
with open('results/before_after_transcript.md', 'w') as f:
    f.write('# SENTINEL — Before / After Behavior Transcript\n\n')
    f.write('```\n' + transcript + '\n```\n')
print('\nTranscript saved to results/before_after_transcript.md')

## Cell 7 — Results Summary

In [ ]:
import json, os

b_mean = sum(baseline_rewards) / len(baseline_rewards)
t_mean = sum(trained_rewards)  / len(trained_rewards)

print('\n' + '='*60)
print('  SENTINEL — Final Results Summary')
print('='*60)
print(f'{"Metric":<30} {"Baseline":>12} {"Trained":>12}')
print('-'*56)
print(f'{"Mean Reward":<30} {b_mean:>12.4f} {t_mean:>12.4f}')
print(f'{"Min Reward":<30} {min(baseline_rewards):>12.4f} {min(trained_rewards):>12.4f}')
print(f'{"Max Reward":<30} {max(baseline_rewards):>12.4f} {max(trained_rewards):>12.4f}')
print('='*56)

# Save to JSON
os.makedirs('results', exist_ok=True)
results = {
    'baseline': {'mean': b_mean, 'min': min(baseline_rewards), 'max': max(baseline_rewards), 'all': baseline_rewards},
    'trained':  {'mean': t_mean, 'min': min(trained_rewards),  'max': max(trained_rewards),  'all': trained_rewards},
}
with open('results/simulation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nResults saved to results/simulation_results.json')
print('Plot saved to results/training_curves.png')
print('Transcript saved to results/before_after_transcript.md')